# Smart Grid Energy Consumption Forecasting & Analysis — Colab runnerThis notebook starts the **real** FastAPI + scikit-learn application (no static mock-ups).Two ways to get the code into Colab:1. **Upload the folder** — zip `smartgrid/`, upload it, then run the unzip cell.2. **Clone from Git** — put the folder in a repo and run the `git clone` cell.Order: (1) install → (2) get code → (3) start server → (4) open tunnel → (5) test the API.

In [ ]:
# 1 — dependencies (about 1 minute on a fresh Colab VM)!pip -q install fastapi "uvicorn[standard]" python-multipart pandas numpy scikit-learn joblib openpyxl reportlab pyngrokprint("dependencies installed")

In [ ]:
# 2a — OPTION A: upload and unzip the project folderfrom google.colab import filesup = files.upload()          # pick smartgrid.zipimport zipfile, osfor name in up:    if name.endswith(".zip"):        with zipfile.ZipFile(name) as z:            z.extractall(".")os.chdir("smartgrid")print(sorted(os.listdir(".")))

In [ ]:
# 2b — OPTION B: clone from Git instead (edit the URL)# !git clone https://github.com/<you>/<repo>.git# %cd repo/smartgrid

In [ ]:
# 3 — sanity check: the three algorithms and the data are presentimport os, pandas as pdprint("engine.py exists:", os.path.exists("engine.py"))print("bundled CEEW files:", [f for f in os.listdir("data") if f.endswith(".csv")])df = pd.read_csv("data/ceew_mathura.csv")print(df.shape, list(df.columns))df.head()

In [ ]:
# 4 — train the three models headlessly (same code path the web app uses)import engine, dbdb.init_db()meta = engine.import_local_file(__import__("pathlib").Path("data/ceew_mathura.csv"),                                "CEEW mathura aggregated.csv")print("dataset id:", meta["id"], "| rows:", meta["rows"])print("detected columns:", meta["detection"]["timestamp_column"], "/",      meta["detection"]["consumption_column"])prep = engine.prepare(meta["id"])print("grain:", prep["grain"], "| continuous rows:", prep["base_rows"])run = engine.train(meta["id"])for name, m in run["metrics"].items():    print(f"{name:20s} MAE={m['MAE']:7.3f}  RMSE={m['RMSE']:7.3f}  R2={m['R2']:.4f}")print("best model:", run["best_model"])

In [ ]:
# 5 — recursive 7-day forecast with the best modelfc = engine.forecast(meta["id"], "best", horizon=7, interval="daily")print(fc["model"], "->", len(fc["points"]), "points, peak", fc["summary"]["peak"])pd.DataFrame(fc["points"])

In [ ]:
# 6 — start the web app (dashboard + API) in the backgroundimport subprocess, time, osos.makedirs("logs", exist_ok=True)proc = subprocess.Popen(    ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000",     "--log-level", "warning"],    stdout=open("logs/server.log", "w"), stderr=subprocess.STDOUT)time.sleep(6)print("running:", proc.poll() is None)print(open("logs/server.log").read()[-800:])

In [ ]:
# 7 — expose it publicly (Colab cannot serve localhost to your browser)from pyngrok import ngrok# ngrok.set_auth_token("PASTE_YOUR_TOKEN")   # free token: https://dashboard.ngrok.compublic = ngrok.connect(8000, bind_tls=True).public_urlprint("Dashboard :", public)print("API docs  :", public + "/docs")

In [ ]:
# 8 — end-to-end API smoke test (runs inside the VM, so localhost is correct here)import requestsbase = "http://localhost:8000"print(requests.get(base + "/health").json())# upload the bundled dataset through the public endpointwith open("data/ceew_mathura.csv", "rb") as fh:    up = requests.post(base + "/upload-dataset/",                       files={"file": ("ceew_mathura.csv", fh, "text/csv")}).json()did = up["dataset"]["id"]print("uploaded:", did, up["detection"])requests.post(base + "/preprocess/", json={"dataset_id": did}).raise_for_status()run = requests.post(base + "/train-models/", json={"dataset_id": did}).json()["run"]print("best:", run["best_model"], "| MAE:", run["metrics"][run["best_model"]]["MAE"])print(requests.post(base + "/predict/",                    json={"dataset_id": did, "horizon": 7, "interval": "daily"}                    ).json()["points"][:2])

In [ ]:
# 9 — live meter endpoint (matches /live-api/readings in the notebook)import requests, random, timebase = "http://localhost:8000"for i in range(5):    requests.post(base + "/live-api/readings", json={        "meter_id": "SMART_METER_001",        "energy_consumption": round(random.uniform(100, 200), 2)}).raise_for_status()    time.sleep(0.4)print(requests.get(base + "/live-api/readings?limit=5").json())print(requests.get(base + "/live-api/meters").json())

In [ ]:
# 10 — stop the server when you are finishedproc.terminate()ngrok.kill()print("server stopped")

## Notes* The Location Explorer downloads the OWID energy dataset (~9 MB) on first use.  If Colab has no outbound network for that host, set a local copy first:  `os.environ["OWID_LOCAL_FILE"] = "/content/owid-energy-data.csv"`.* Annual country data usually **cannot** train three time-series models (too few  rows). The API returns a clear refusal instead of a fabricated metric — that is  intended behaviour, and easy to demonstrate in a viva.* Everything persists inside the Colab VM only. Re-run cells 4–6 after a runtime  restart.